# Modelo ML: Predicción de Emergencias por Hora
**CBT Talcahuano** — Bomberos de Talcahuano

Objetivo: predecir la **cantidad de emergencias por hora** a partir de variables temporales, climáticas y de historial.

Pipeline:
1. Carga y agregación horaria de `tweets_procesados.csv`
2. Descarga de datos climáticos (Open-Meteo)
3. Integración de feriados
4. Feature engineering (temporales + lag + clima)
5. Entrenamiento: Poisson GLM, Random Forest, XGBoost
6. Evaluación y selección del mejor modelo

In [ ]:
import asyncio
import sys

# En Windows con nbconvert, usar SelectorEventLoop para evitar el warning de zmq
if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())

# Backend no-interactivo ANTES de importar pyplot (evita crash por GUI en nbconvert)
import matplotlib
matplotlib.use('Agg')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import requests
import warnings
import joblib
import os

from sklearn.linear_model import PoissonRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (14, 5)
plt.style.use('seaborn-v0_8-whitegrid')

try:
    import xgboost as xgb
    HAS_XGB = True
    print(f"XGBoost {xgb.__version__} disponible")
except ImportError:
    HAS_XGB = False
    print("XGBoost no disponible. Instalar con: pip install xgboost")

print("Imports OK")

---
## 1. Carga y agregación horaria

In [ ]:
# Cargar tweets procesados
tweets = pd.read_csv('../02_data/tweets_procesados.csv', sep=';')

# Timestamp en 'Fecha' está en UTC
tweets['Fecha'] = pd.to_datetime(tweets['Fecha'], utc=True)

# Floor en UTC (sin ambigüedad DST) y luego convertir a Santiago
tweets['fecha_hora'] = tweets['Fecha'].dt.floor('h').dt.tz_convert('America/Santiago')
tweets['Fecha_local'] = tweets['Fecha'].dt.tz_convert('America/Santiago')

print(f"Rango de datos: {tweets['Fecha_local'].min()} → {tweets['Fecha_local'].max()}")
print(f"Total registros: {len(tweets):,}")
tweets.head(3)

In [ ]:
# Contar emergencias por hora (variable objetivo)
emergencias_h = (
    tweets.groupby('fecha_hora')
    .size()
    .rename('n_emergencias')
    .reset_index()
)

# Crear índice horario completo sin gaps (horas con 0 emergencias = datos reales)
idx_completo = pd.date_range(
    start=emergencias_h['fecha_hora'].min().floor('D'),
    end=emergencias_h['fecha_hora'].max().ceil('D'),
    freq='h',
    tz='America/Santiago'
)

df = (
    pd.DataFrame({'fecha_hora': idx_completo})
    .merge(emergencias_h, on='fecha_hora', how='left')
    .fillna({'n_emergencias': 0})
)
df['n_emergencias'] = df['n_emergencias'].astype(int)

print(f"Total horas en índice: {len(df):,}")
print(f"Horas con ≥1 emergencia: {(df['n_emergencias'] > 0).sum():,} ({(df['n_emergencias'] > 0).mean()*100:.1f}%)")
print(f"Media emergencias/hora:  {df['n_emergencias'].mean():.4f}")
print(f"Max emergencias/hora:    {df['n_emergencias'].max()}")
df.head(6)

---
## 2. Variables climáticas (Open-Meteo Archive API)

In [ ]:
def fetch_weather(start_date: str, end_date: str,
                  lat: float = -36.731106, lon: float = -73.11023) -> pd.DataFrame:
    """Descarga datos climáticos históricos por hora desde Open-Meteo."""
    url = (
        f"https://archive-api.open-meteo.com/v1/archive"
        f"?latitude={lat}&longitude={lon}"
        f"&start_date={start_date}&end_date={end_date}"
        f"&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m,"
        f"wind_gusts_10m,precipitation,weather_code"
        f"&timezone=America%2FSantiago&format=csv"
    )
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    from io import StringIO
    df_w = pd.read_csv(StringIO(resp.text), skiprows=3)
    df_w.columns = ['fecha_hora', 'temperatura', 'humedad', 'viento_vel',
                    'viento_racha', 'precipitacion', 'cod_clima']
    # ambiguous/nonexistent='NaT' para manejar cambios de horario DST
    df_w['fecha_hora'] = pd.to_datetime(df_w['fecha_hora']).dt.tz_localize(
        'America/Santiago', ambiguous='NaT', nonexistent='NaT'
    )
    df_w = df_w.dropna(subset=['fecha_hora'])
    return df_w


# Rango: desde inicio de datos hasta hace 5 días (la API archive tiene lag)
fecha_ini = df['fecha_hora'].min().strftime('%Y-%m-%d')
fecha_fin = (pd.Timestamp.now(tz='America/Santiago') - pd.Timedelta(days=5)).strftime('%Y-%m-%d')

print(f"Descargando clima: {fecha_ini} → {fecha_fin} ...")
try:
    weather = fetch_weather(fecha_ini, fecha_fin)
    print(f"✓ Datos climáticos descargados: {len(weather):,} horas")
    display(weather.head(3))
except Exception as e:
    weather = pd.DataFrame(columns=['fecha_hora', 'temperatura', 'humedad', 'viento_vel',
                                    'viento_racha', 'precipitacion', 'cod_clima'])
    print(f"✗ Error al descargar clima: {e}")
    print("  El modelo se entrenará solo con variables temporales y de lag.")

---
## 3. Feriados

In [ ]:
import re

# El archivo tiene una fila de título antes del header real
feriados_df = pd.read_excel('../02_data/independent_variables/feriados.xlsx', header=1)
print("Columnas:", feriados_df.columns.tolist())
print(feriados_df.head(5))

# Los feriados tienen formato de texto "Miércoles, 01 de Enero" (sin año)
# Los expandimos para todos los años del dataset
MESES_ES = {
    'enero':1,'febrero':2,'marzo':3,'abril':4,'mayo':5,'junio':6,
    'julio':7,'agosto':8,'septiembre':9,'octubre':10,'noviembre':11,'diciembre':12
}

fecha_col = feriados_df.columns[0]  # Primera columna = "Día"
fechas_feriado = set()
años = range(2021, 2027)

for texto in feriados_df[fecha_col].dropna():
    m = re.search(r'(\d{1,2})\s+de\s+(\w+)', str(texto).lower())
    if m:
        dia = int(m.group(1))
        mes = MESES_ES.get(m.group(2).strip())
        if mes:
            for año in años:
                try:
                    fechas_feriado.add(pd.Timestamp(año, mes, dia).date())
                except ValueError:
                    pass

print(f"\nTotal días feriados cargados (expandidos por año): {len(fechas_feriado)}")

---
## 4. Feature Engineering

In [ ]:
def agregar_features_temporales(df: pd.DataFrame) -> pd.DataFrame:
    dt = df['fecha_hora']

    # Variables temporales directas
    df['hora']        = dt.dt.hour
    df['dia_semana']  = dt.dt.dayofweek   # 0=Lun, 6=Dom
    df['mes']         = dt.dt.month
    df['dia_mes']     = dt.dt.day
    df['año']         = dt.dt.year
    df['trimestre']   = dt.dt.quarter
    df['semana_año']  = dt.dt.isocalendar().week.astype(int)

    # Indicadores binarios
    df['es_fin_semana'] = (df['dia_semana'] >= 5).astype(int)
    df['es_verano']     = df['mes'].isin([12, 1, 2]).astype(int)  # hemisferio sur
    df['es_feriado']    = dt.dt.date.map(lambda d: int(d in fechas_feriado))
    df['es_noche']      = ((df['hora'] >= 22) | (df['hora'] < 6)).astype(int)

    # Codificación cíclica (para capturar periodicidad sin romper la métrica de distancia)
    df['hora_sin']      = np.sin(2 * np.pi * df['hora'] / 24)
    df['hora_cos']      = np.cos(2 * np.pi * df['hora'] / 24)
    df['dia_sem_sin']   = np.sin(2 * np.pi * df['dia_semana'] / 7)
    df['dia_sem_cos']   = np.cos(2 * np.pi * df['dia_semana'] / 7)
    df['mes_sin']       = np.sin(2 * np.pi * df['mes'] / 12)
    df['mes_cos']       = np.cos(2 * np.pi * df['mes'] / 12)

    return df


df = agregar_features_temporales(df)
print("Features temporales creadas")

In [ ]:
# Merge con datos climáticos
COLS_CLIMA = ['temperatura', 'humedad', 'viento_vel', 'viento_racha', 'precipitacion', 'cod_clima']

if len(weather) > 0:
    df = df.merge(weather.rename(columns={'fecha_hora': 'fecha_hora_w'})
                           .assign(fecha_hora=lambda x: x['fecha_hora_w'])[['fecha_hora'] + COLS_CLIMA],
                  on='fecha_hora', how='left')
    # Imputar NaN con mediana por hora del día (para horas fuera del rango de la API)
    for col in COLS_CLIMA:
        df[col] = df.groupby('hora')[col].transform(lambda x: x.fillna(x.median()))
    pct_nans = df[COLS_CLIMA].isna().mean().round(3)
    print("NaN restantes por columna:")
    print(pct_nans[pct_nans > 0].to_string() or "  Ninguno")
else:
    for col in COLS_CLIMA:
        df[col] = np.nan
    print("Sin datos climáticos — las columnas quedarán como NaN y se excluirán del modelo.")

In [ ]:
# Features de lag (usar solo valores pasados → sin data leakage)
df = df.sort_values('fecha_hora').reset_index(drop=True)

n = df['n_emergencias']
df['lag_1h']          = n.shift(1)
df['lag_2h']          = n.shift(2)
df['lag_3h']          = n.shift(3)
df['lag_24h']         = n.shift(24)
df['lag_48h']         = n.shift(48)
df['lag_168h']        = n.shift(168)   # misma hora la semana pasada

# Estadísticas móviles (ventana hacia atrás; shift(1) para no incluir la hora actual)
df['roll_mean_24h']   = n.shift(1).rolling(24,  min_periods=12).mean()
df['roll_mean_7d']    = n.shift(1).rolling(168, min_periods=48).mean()
df['roll_std_7d']     = n.shift(1).rolling(168, min_periods=48).std()
df['roll_sum_24h']    = n.shift(1).rolling(24,  min_periods=12).sum()
df['roll_max_24h']    = n.shift(1).rolling(24,  min_periods=12).max()

print("Features de lag creadas")

---
## 5. Análisis exploratorio (EDA)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Patrones de Emergencias — CBT Talcahuano', fontsize=14, fontweight='bold')

# Por hora del día
ax = axes[0, 0]
df.groupby('hora')['n_emergencias'].mean().plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Promedio por hora del día')
ax.set_xlabel('Hora')
ax.set_ylabel('Emergencias promedio')
ax.tick_params(axis='x', rotation=0)

# Por día de la semana
ax = axes[0, 1]
dias = ['Lun', 'Mar', 'Mié', 'Jue', 'Vie', 'Sáb', 'Dom']
medias = df.groupby('dia_semana')['n_emergencias'].mean()
medias.plot(kind='bar', ax=ax, color='coral', edgecolor='white')
ax.set_xticklabels(dias, rotation=0)
ax.set_title('Promedio por día de la semana')
ax.set_xlabel('')
ax.set_ylabel('Emergencias promedio')

# Por mes
ax = axes[0, 2]
meses = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']
df.groupby('mes')['n_emergencias'].mean().plot(kind='bar', ax=ax, color='mediumseagreen', edgecolor='white')
ax.set_xticklabels(meses, rotation=45, ha='right')
ax.set_title('Promedio por mes')
ax.set_xlabel('')
ax.set_ylabel('Emergencias promedio')

# Distribución de conteos por hora
ax = axes[1, 0]
vc = df['n_emergencias'].value_counts().sort_index()
vc.plot(kind='bar', ax=ax, color='mediumpurple', edgecolor='white')
ax.set_title('Distribución emergencias/hora')
ax.set_xlabel('Nº emergencias en una hora')
ax.set_ylabel('Frecuencia (horas)')
ax.tick_params(axis='x', rotation=0)
# Anotar % de horas con 0
pct_cero = (df['n_emergencias'] == 0).mean() * 100
ax.text(0.65, 0.85, f'{pct_cero:.0f}% horas\nsin emergencias',
        transform=ax.transAxes, fontsize=9, color='gray')

# Serie temporal mensual
ax = axes[1, 1]
df.set_index('fecha_hora').resample('ME')['n_emergencias'].sum().plot(ax=ax, color='darkorange', linewidth=2)
ax.set_title('Total mensual de emergencias')
ax.set_xlabel('')
ax.set_ylabel('Total emergencias')
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')

# Heatmap hora × día de semana
ax = axes[1, 2]
pivot = df.pivot_table(values='n_emergencias', index='hora', columns='dia_semana', aggfunc='mean')
pivot.columns = dias
sns.heatmap(pivot, ax=ax, cmap='YlOrRd', cbar_kws={'label': 'Promedio'}, linewidths=0.2)
ax.set_title('Heatmap: hora × día de semana')
ax.set_ylabel('Hora del día')

plt.tight_layout()
plt.savefig('eda_emergencias.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figura guardada: eda_emergencias.png")

---
## 6. Preparar dataset para ML

In [ ]:
FEATURES_TEMPORALES = [
    'hora', 'dia_semana', 'mes', 'dia_mes', 'año', 'trimestre', 'semana_año',
    'es_fin_semana', 'es_verano', 'es_feriado', 'es_noche',
    'hora_sin', 'hora_cos', 'dia_sem_sin', 'dia_sem_cos', 'mes_sin', 'mes_cos',
]
FEATURES_CLIMA = ['temperatura', 'humedad', 'viento_vel', 'viento_racha', 'precipitacion', 'cod_clima']
FEATURES_LAG   = [
    'lag_1h', 'lag_2h', 'lag_3h', 'lag_24h', 'lag_48h', 'lag_168h',
    'roll_mean_24h', 'roll_mean_7d', 'roll_std_7d', 'roll_sum_24h', 'roll_max_24h',
]
TARGET = 'n_emergencias'

# Excluir features que tienen más del 70% de NaN
all_features = FEATURES_TEMPORALES + FEATURES_CLIMA + FEATURES_LAG
features = [f for f in all_features
            if f in df.columns and df[f].notna().mean() > 0.30]

print(f"Features incluidas ({len(features)}):")
for f in features:
    pct = df[f].isna().mean() * 100
    tag = "[CLIMA]" if f in FEATURES_CLIMA else ("[LAG]" if f in FEATURES_LAG else "[TEMP]")
    print(f"  {tag:7s} {f:<22s}  {pct:.1f}% NaN")

In [ ]:
# Dataset limpio (eliminar filas con NaN en features)
df_ml = df[['fecha_hora', TARGET] + features].dropna()

print(f"Filas totales:           {len(df):,}")
print(f"Filas después de dropna: {len(df_ml):,}  ({len(df_ml)/len(df)*100:.1f}%)")

# Split temporal 80/20 (respetar el orden cronológico — no shuffle)
split = int(len(df_ml) * 0.80)
df_train = df_ml.iloc[:split]
df_test  = df_ml.iloc[split:]

X_train = df_train[features]
y_train = df_train[TARGET]
X_test  = df_test[features]
y_test  = df_test[TARGET]

print(f"\nTrain: {len(X_train):,} horas  "
      f"({df_train['fecha_hora'].min().date()} → {df_train['fecha_hora'].max().date()})")
print(f"Test:  {len(X_test):,} horas  "
      f"({df_test['fecha_hora'].min().date()} → {df_test['fecha_hora'].max().date()})")
print(f"\nMedia emerg/hora — train: {y_train.mean():.4f}  |  test: {y_test.mean():.4f}")

---
## 7. Entrenamiento de modelos

In [ ]:
def evaluar(nombre, y_true, y_pred, resultados_lst):
    y_pred = np.clip(y_pred, 0, None)  # los conteos no pueden ser negativos
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    # Pseudo R²
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot
    # Poisson deviance (métrica natural para conteos)
    eps = 1e-8
    dev = 2 * np.mean(
        np.where(y_true > 0, y_true * np.log((y_true + eps) / (y_pred + eps)), 0)
        - (y_true - y_pred)
    )
    print(f"\n── {nombre} ──")
    print(f"  MAE:              {mae:.4f}")
    print(f"  RMSE:             {rmse:.4f}")
    print(f"  Poisson deviance: {dev:.4f}")
    print(f"  R²:               {r2:.4f}")
    resultados_lst.append({'modelo': nombre, 'MAE': mae, 'RMSE': rmse, 'Poisson_Dev': dev, 'R2': r2})
    return y_pred

resultados = []
predicciones = {}

In [ ]:
# ── Modelo 1: Regresión de Poisson (baseline lineal) ──────────────────
scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_te_sc = scaler.transform(X_test)

poisson = PoissonRegressor(alpha=0.1, max_iter=2000)
poisson.fit(X_tr_sc, y_train)
predicciones['Poisson GLM'] = evaluar('Poisson GLM', y_test, poisson.predict(X_te_sc), resultados)

In [ ]:
# ── Modelo 2: Random Forest ───────────────────────────────────────────
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=5,
    max_features='sqrt',
    n_jobs=-1,
    random_state=42,
)
rf.fit(X_train, y_train)
predicciones['Random Forest'] = evaluar('Random Forest', y_test, rf.predict(X_test), resultados)

In [ ]:
# ── Modelo 3: XGBoost con objetivo Poisson ────────────────────────────
if HAS_XGB:
    xgb_model = xgb.XGBRegressor(
        objective='count:poisson',
        n_estimators=800,
        max_depth=6,
        learning_rate=0.04,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=5,
        reg_alpha=0.1,
        reg_lambda=1.0,
        n_jobs=-1,
        random_state=42,
        early_stopping_rounds=50,
        eval_metric='poisson-nloglik',
    )
    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=100,
    )
    predicciones['XGBoost (Poisson)'] = evaluar(
        'XGBoost (Poisson)', y_test, xgb_model.predict(X_test), resultados
    )
else:
    print("XGBoost no instalado — saltar este modelo.")

---
## 8. Comparación y visualización

In [ ]:
df_res = pd.DataFrame(resultados)
print("\n" + "="*55)
print("RESUMEN DE RESULTADOS")
print("="*55)
print(df_res.to_string(index=False))
mejor = df_res.loc[df_res['MAE'].idxmin(), 'modelo']
print(f"\n→ Mejor modelo (menor MAE): {mejor}")

In [ ]:
# Elegir el mejor modelo para visualizar
y_pred_best = predicciones[mejor]
n_viz = 24 * 14  # 2 semanas

fechas_test = df_test['fecha_hora'].values[:n_viz]
y_real_viz  = y_test.values[:n_viz]
y_pred_viz  = y_pred_best[:n_viz]

fig, axes = plt.subplots(2, 1, figsize=(16, 9))
fig.suptitle(f'Emergencias reales vs predichas — {mejor}', fontsize=13, fontweight='bold')

# Serie temporal — 2 semanas del test
ax = axes[0]
fechas_pd = pd.to_datetime(fechas_test)
ax.fill_between(fechas_pd, 0, y_real_viz, alpha=0.4, label='Real', color='steelblue')
ax.plot(fechas_pd, y_pred_viz, color='crimson', linewidth=1.5,
        label=f'Predicción ({mejor})', alpha=0.85)
ax.set_ylabel('Nº emergencias')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m %Hh'))
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')

# Scatter real vs predicho (todo el test)
ax = axes[1]
ax.scatter(y_test, y_pred_best, alpha=0.15, s=12, color='darkorange', label='Horas')
lim = max(y_test.max(), y_pred_best.max()) + 0.5
ax.plot([0, lim], [0, lim], 'k--', linewidth=1, label='Perfecto')
ax.set_xlabel('Real')
ax.set_ylabel('Predicho')
ax.set_title('Scatter: real vs predicho (conjunto test completo)')
ax.legend()

plt.tight_layout()
plt.savefig('predicciones.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figura guardada: predicciones.png")

In [ ]:
# Importancia de features del mejor modelo
if mejor.startswith('XGBoost') and HAS_XGB:
    importancias = pd.Series(xgb_model.feature_importances_, index=features)
else:
    importancias = pd.Series(rf.feature_importances_, index=features)

importancias = importancias.sort_values(ascending=True)

plt.figure(figsize=(10, max(6, len(features) * 0.3)))
colors = ['steelblue' if f in FEATURES_LAG else
          ('darkorange' if f in FEATURES_CLIMA else 'mediumseagreen')
          for f in importancias.index]
importancias.plot(kind='barh', color=colors, edgecolor='white')
plt.title(f'Importancia de features — {mejor}', fontweight='bold')
plt.xlabel('Importancia')

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='steelblue',      label='Lag / ventanas móviles'),
    Patch(facecolor='darkorange',     label='Climáticas'),
    Patch(facecolor='mediumseagreen', label='Temporales'),
]
plt.legend(handles=legend_elements, loc='lower right')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figura guardada: feature_importance.png")

---
## 9. Guardar modelo y artefactos

In [ ]:
os.makedirs('modelos', exist_ok=True)

# Guardar el mejor modelo
if mejor.startswith('XGBoost') and HAS_XGB:
    xgb_model.save_model('modelos/xgb_emergencias_hora.json')
    print("✓ XGBoost guardado: modelos/xgb_emergencias_hora.json")

joblib.dump(rf,                    'modelos/rf_emergencias_hora.pkl')
joblib.dump(scaler,                'modelos/scaler_poisson.pkl')
joblib.dump(features,              'modelos/features.pkl')
joblib.dump(mejor,                 'modelos/mejor_modelo.pkl')

print("✓ Random Forest guardado:  modelos/rf_emergencias_hora.pkl")
print("✓ Scaler guardado:         modelos/scaler_poisson.pkl")
print("✓ Lista de features:       modelos/features.pkl")

# Guardar dataset final para reproducibilidad
df_ml.to_csv('dataset_final_ml.csv', index=False)
print("✓ Dataset final guardado:  dataset_final_ml.csv")

# Guardar tabla de resultados
df_res.to_csv('resultados_modelos.csv', index=False)
print("✓ Resultados guardados:    resultados_modelos.csv")

---
## 10. Predicción de ejemplo

Cómo usar el modelo para predecir la cantidad esperada de emergencias en una hora específica.

In [ ]:
def predecir_hora(fecha_hora_local: str, historial: pd.DataFrame = None) -> dict:
    """
    Predice la cantidad esperada de emergencias para una hora dada.

    Args:
        fecha_hora_local: ISO string, e.g. '2025-07-15 14:00:00'
        historial: DataFrame con columna 'n_emergencias' indexado por hora (para lag features)

    Returns:
        dict con 'prediccion' (float) y 'probabilidad_al_menos_1' (float)
    """
    ts = pd.Timestamp(fecha_hora_local, tz='America/Santiago')

    row = {
        'hora':         ts.hour,
        'dia_semana':   ts.dayofweek,
        'mes':          ts.month,
        'dia_mes':      ts.day,
        'año':          ts.year,
        'trimestre':    ts.quarter,
        'semana_año':   ts.isocalendar()[1],
        'es_fin_semana': int(ts.dayofweek >= 5),
        'es_verano':    int(ts.month in [12, 1, 2]),
        'es_feriado':   int(ts.date() in fechas_feriado),
        'es_noche':     int(ts.hour >= 22 or ts.hour < 6),
        'hora_sin':     np.sin(2 * np.pi * ts.hour / 24),
        'hora_cos':     np.cos(2 * np.pi * ts.hour / 24),
        'dia_sem_sin':  np.sin(2 * np.pi * ts.dayofweek / 7),
        'dia_sem_cos':  np.cos(2 * np.pi * ts.dayofweek / 7),
        'mes_sin':      np.sin(2 * np.pi * ts.month / 12),
        'mes_cos':      np.cos(2 * np.pi * ts.month / 12),
    }

    # Lag features: usar historial si está disponible
    if historial is not None:
        h = historial
        def lag(n): return h.get(ts - pd.Timedelta(hours=n), np.nan)
        row.update({
            'lag_1h': lag(1), 'lag_2h': lag(2), 'lag_3h': lag(3),
            'lag_24h': lag(24), 'lag_48h': lag(48), 'lag_168h': lag(168),
        })
    else:
        # Usar medias históricas como proxy cuando no hay historial
        medias_hora = df_ml.groupby('hora')['n_emergencias'].mean()
        media_h = medias_hora.get(ts.hour, df_ml['n_emergencias'].mean())
        for k in ['lag_1h','lag_2h','lag_3h','lag_24h','lag_48h','lag_168h']:
            row[k] = media_h
        row.update({'roll_mean_24h': media_h, 'roll_mean_7d': media_h,
                    'roll_std_7d': 0, 'roll_sum_24h': media_h * 24,
                    'roll_max_24h': media_h * 3})

    X = pd.DataFrame([{f: row.get(f, np.nan) for f in features}])

    if mejor.startswith('XGBoost') and HAS_XGB:
        pred = float(np.clip(xgb_model.predict(X)[0], 0, None))
    else:
        pred = float(np.clip(rf.predict(X)[0], 0, None))

    # Probabilidad de al menos 1 emergencia (distribución de Poisson con λ=pred)
    from scipy.stats import poisson
    prob_al_menos_1 = 1 - poisson.pmf(0, mu=pred)

    return {
        'fecha_hora': str(ts),
        'emergencias_esperadas': round(pred, 3),
        'probabilidad_al_menos_1': round(prob_al_menos_1, 3),
        'modelo': mejor,
    }


# Ejemplo de uso
ejemplo = predecir_hora('2025-07-15 14:00:00')
print("Predicción de ejemplo:")
for k, v in ejemplo.items():
    print(f"  {k}: {v}")